In [1]:
# Load source data and verified Phase 10/ML evidence without modifying the original CSV

from pathlib import Path
import json
import hashlib
import warnings
import numpy as np
import pandas as pd

PROJECT_ROOT = Path(r"C:\Users\hp\Nassau_Candy_Final_Submission")
RAW_FILE = PROJECT_ROOT / "data" / "raw" / "Nassau Candy Distributor.csv"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
OPTIMIZATION_DIR = PROJECT_ROOT / "reports" / "optimization_recommendations"
OUTPUT_DIR = PROJECT_ROOT / "reports" / "kpi_risk_analysis"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

EXPECTED_SOURCE_SHA256 = (
    "6ae9208e77ce410e90073b19813d29f89853ce9f3cce43d41f9643781696e94f"
)

required_source_columns = [
    "Row ID", "Order ID", "Order Date", "Ship Date", "Ship Mode",
    "Customer ID", "Country/Region", "City", "State/Province",
    "Postal Code", "Division", "Region", "Product ID",
    "Product Name", "Sales", "Units", "Gross Profit", "Cost"
]

required_files = {
    "model_metadata": PROJECT_ROOT / "models" / "Nassau_Lead_Time_Model_Metadata.json",
    "model_comparison": PROCESSED_DIR / "ML_Final_Model_Comparison.csv",
    "cross_validation": PROCESSED_DIR / "ML_Cross_Validation.csv",
    "predictions": PROCESSED_DIR / "ML_Predictions.csv",
    "lead_time_ranges": PROCESSED_DIR / "ML_LeadTime_Ranges.csv",
    "margin_stability": PROCESSED_DIR / "margin_stability_proxy.csv",
    "profit_variability": PROCESSED_DIR / "profit_variability_report.csv",
    "phase10_coverage": OPTIMIZATION_DIR / "task_10_17_coverage_kpis.csv",
    "phase10_screening": OPTIMIZATION_DIR / "Phase_10_Factory_Option_Screening.csv",
    "phase10_unsupported": OPTIMIZATION_DIR / "task_10_18_unsupported_scenarios.csv",
    "phase10_lead_time": OPTIMIZATION_DIR / "task_10_04_lead_time_comparison.csv",
}

missing_files = [name for name, path in required_files.items() if not path.exists()]
if not RAW_FILE.exists():
    raise FileNotFoundError(f"Original source CSV not found: {RAW_FILE}")
if missing_files:
    raise FileNotFoundError(f"Required project files are missing: {missing_files}")

source_hash_before = hashlib.sha256(RAW_FILE.read_bytes()).hexdigest()
if source_hash_before != EXPECTED_SOURCE_SHA256:
    warnings.warn(
        "Source hash differs from the previously verified project hash. "
        "Do not overwrite or alter the source file; investigate before submission."
    )

# Read the source into a working DataFrame only
df = pd.read_csv(RAW_FILE)

missing_columns = sorted(set(required_source_columns) - set(df.columns))
if missing_columns:
    raise ValueError(f"Source CSV is missing required columns: {missing_columns}")

if len(df) != 10194:
    warnings.warn(f"Expected 10,194 source rows; found {len(df):,}.")

# Preserve original date strings in df; parsed dates and lead time live in separate columns
analysis_df = df.copy()
analysis_df["Order_Date_Parsed"] = pd.to_datetime(
    analysis_df["Order Date"], dayfirst=True, errors="coerce"
)
analysis_df["Ship_Date_Parsed"] = pd.to_datetime(
    analysis_df["Ship Date"], dayfirst=True, errors="coerce"
)
analysis_df["Derived_Lead_Time_Days"] = (
    analysis_df["Ship_Date_Parsed"] - analysis_df["Order_Date_Parsed"]
).dt.days

for col in ["Sales", "Units", "Gross Profit", "Cost"]:
    analysis_df[col] = pd.to_numeric(analysis_df[col], errors="coerce")

# Common output paths and reusable product/region aggregates
product_agg = (
    analysis_df.groupby("Product Name", dropna=False)
    .agg(
        Order_Lines=("Row ID", "count"),
        Unique_Orders=("Order ID", "nunique"),
        Total_Sales=("Sales", "sum"),
        Total_Gross_Profit=("Gross Profit", "sum"),
        Total_Cost=("Cost", "sum"),
        Total_Units=("Units", "sum"),
        Mean_Lead_Time_Days=("Derived_Lead_Time_Days", "mean"),
        Median_Lead_Time_Days=("Derived_Lead_Time_Days", "median"),
        Lead_Time_Std_Days=("Derived_Lead_Time_Days", "std"),
        Lead_Time_IQR_Days=(
            "Derived_Lead_Time_Days",
            lambda x: x.quantile(0.75) - x.quantile(0.25)
        ),
    )
    .reset_index()
)

product_agg["Gross_Margin_Pct"] = np.where(
    product_agg["Total_Sales"].ne(0),
    product_agg["Total_Gross_Profit"] / product_agg["Total_Sales"] * 100,
    np.nan
)

region_agg = (
    analysis_df.groupby("Region", dropna=False)
    .agg(
        Order_Lines=("Row ID", "count"),
        Unique_Orders=("Order ID", "nunique"),
        Total_Sales=("Sales", "sum"),
        Total_Gross_Profit=("Gross Profit", "sum"),
        Total_Units=("Units", "sum"),
        Mean_Lead_Time_Days=("Derived_Lead_Time_Days", "mean"),
        Median_Lead_Time_Days=("Derived_Lead_Time_Days", "median"),
        Lead_Time_Std_Days=("Derived_Lead_Time_Days", "std"),
    )
    .reset_index()
)

region_agg["Gross_Margin_Pct"] = np.where(
    region_agg["Total_Sales"].ne(0),
    region_agg["Total_Gross_Profit"] / region_agg["Total_Sales"] * 100,
    np.nan
)

# Load verified supporting files
support = {
    name: pd.read_csv(path)
    for name, path in required_files.items()
    if path.suffix.lower() == ".csv"
}
with open(required_files["model_metadata"], "r", encoding="utf-8") as f:
    model_metadata = json.load(f)

print("Source rows:", len(df))
print("Source columns:", len(df.columns))
print("Source SHA-256:", source_hash_before)
print("Product groups:", product_agg["Product Name"].nunique())
print("Region groups:", region_agg["Region"].nunique())
print("Output folder:", OUTPUT_DIR)

Source rows: 10194
Source columns: 18
Source SHA-256: 6ae9208e77ce410e90073b19813d29f89853ce9f3cce43d41f9643781696e94f
Product groups: 15
Region groups: 4
Output folder: C:\Users\hp\Nassau_Candy_Final_Submission\reports\kpi_risk_analysis


#Total sales KPI

In [4]:
# Total sales KPI
# Calculate total historical sales from the original source values

total_sales = analysis_df["Sales"].sum(min_count=1)

kpi_total_sales = pd.DataFrame([{
    "KPI": "Total Sales",
    "Value": total_sales,
    "Unit": "Source currency units",
    "Scope": "All source order lines",
    "Record_Count": int(analysis_df["Sales"].count()),
    "Status": "Calculated from original source",
    "Interpretation": "Sum of historical Sales values; not a forecast."
}])

display(kpi_total_sales)

,KPI,Value,Unit,Scope,Record_Count,Status,Interpretation
0,Total Sales,141783.63,Source currency units,All source order lines,10194,Calculated from original source,Sum of historical Sales values; not a forecast.


In [6]:
# Gross profit KPI

# Calculate gross profit using the source Gross Profit field

total_gross_profit = analysis_df["Gross Profit"].sum(min_count=1)

kpi_gross_profit = pd.DataFrame([{
    "KPI": "Total Gross Profit",
    "Value": total_gross_profit,
    "Unit": "Source currency units",
    "Scope": "All source order lines",
    "Record_Count": int(analysis_df["Gross Profit"].count()),
    "Status": "Calculated from source Gross Profit column",
    "Interpretation": (
        "Uses the recorded Gross Profit field; it is not silently replaced "
        "with a Sales-minus-Cost calculation."
    )
}])

display(kpi_gross_profit)

,KPI,Value,Unit,Scope,Record_Count,Status,Interpretation
0,Total Gross Profit,93442.8,Source currency units,All source order lines,10194,Calculated from source Gross Profit column,Uses the recorded Gross Profit field; it is no...


In [8]:
# Profit margin KPI

# Calculate overall weighted gross margin with safe division

if pd.notna(total_sales) and total_sales != 0:
    overall_margin_pct = (total_gross_profit / total_sales) * 100
else:
    overall_margin_pct = np.nan

kpi_profit_margin = pd.DataFrame([{
    "KPI": "Overall Gross Profit Margin",
    "Value": overall_margin_pct,
    "Unit": "Percent",
    "Scope": "Total Gross Profit divided by Total Sales",
    "Status": "Calculated" if pd.notna(overall_margin_pct) else "Not calculable: zero or missing sales",
    "Interpretation": (
        "Weighted overall margin. This is not the simple average of row-level margins."
    )
}])

display(kpi_profit_margin)

,KPI,Value,Unit,Scope,Status,Interpretation
0,Overall Gross Profit Margin,65.905211,Percent,Total Gross Profit divided by Total Sales,Calculated,Weighted overall margin. This is not the simpl...


In [10]:
# Average derived lead-time KPI

# Calculate the average Order Date-to-Ship Date difference

valid_lead_times = analysis_df["Derived_Lead_Time_Days"].dropna()
average_derived_lead_time = (
    valid_lead_times.mean() if not valid_lead_times.empty else np.nan
)

kpi_average_lead_time = pd.DataFrame([{
    "KPI": "Average Derived Lead Time",
    "Value": average_derived_lead_time,
    "Unit": "Days",
    "Scope": "Rows with both dates successfully parsed",
    "Valid_Record_Count": int(valid_lead_times.count()),
    "Status": "Historical date difference",
    "Interpretation": (
        "Order Date-to-Ship Date difference only. The unusually long observed "
        "values mean this must not be described as verified transit duration."
    )
}])

display(kpi_average_lead_time)

,KPI,Value,Unit,Scope,Valid_Record_Count,Status,Interpretation
0,Average Derived Lead Time,1320.841868,Days,Rows with both dates successfully parsed,10194,Historical date difference,Order Date-to-Ship Date difference only. The u...


In [12]:
# Median derived lead-time KPI

# Calculate the median Order Date-to-Ship Date difference

median_derived_lead_time = (
    valid_lead_times.median() if not valid_lead_times.empty else np.nan
)

kpi_median_lead_time = pd.DataFrame([{
    "KPI": "Median Derived Lead Time",
    "Value": median_derived_lead_time,
    "Unit": "Days",
    "Scope": "Rows with both dates successfully parsed",
    "Valid_Record_Count": int(valid_lead_times.count()),
    "Status": "Historical date difference",
    "Interpretation": (
        "Median of the derived date differences; not a verified factory or "
        "transport transit-time measure."
    )
}])

display(kpi_median_lead_time)

,KPI,Value,Unit,Scope,Valid_Record_Count,Status,Interpretation
0,Median Derived Lead Time,1274.0,Days,Rows with both dates successfully parsed,10194,Historical date difference,Median of the derived date differences; not a ...


In [14]:
# Lead-time variability KPI

# Calculate standard deviation and interquartile range of derived lead-time days

lead_time_std = (
    valid_lead_times.std(ddof=1) if len(valid_lead_times) > 1 else np.nan
)
lead_time_q1 = valid_lead_times.quantile(0.25) if len(valid_lead_times) else np.nan
lead_time_q3 = valid_lead_times.quantile(0.75) if len(valid_lead_times) else np.nan
lead_time_iqr = (
    lead_time_q3 - lead_time_q1
    if pd.notna(lead_time_q1) and pd.notna(lead_time_q3)
    else np.nan
)

kpi_lead_time_variability = pd.DataFrame([
    {
        "KPI": "Derived Lead-Time Standard Deviation",
        "Value": lead_time_std,
        "Unit": "Days",
        "Method": "Sample standard deviation (ddof=1)",
        "Interpretation": "Descriptive variability of historical derived date differences."
    },
    {
        "KPI": "Derived Lead-Time Interquartile Range",
        "Value": lead_time_iqr,
        "Unit": "Days",
        "Method": "75th percentile minus 25th percentile",
        "Interpretation": "Spread of the middle 50% of historical derived date differences."
    }
])

display(kpi_lead_time_variability)

,KPI,Value,Unit,Method,Interpretation
0,Derived Lead-Time Standard Deviation,262.444892,Days,Sample standard deviation (ddof=1),Descriptive variability of historical derived ...
1,Derived Lead-Time Interquartile Range,367.000000,Days,75th percentile minus 25th percentile,Spread of the middle 50% of historical derived...


In [16]:
# Estimated lead-time reduction

# Check whether Phase 10 supports any factory-specific lead-time reduction estimate

phase10_lead_time = support["phase10_lead_time"].copy()

estimable_col = "Factory_Specific_Lead_Time_Estimable"
if estimable_col not in phase10_lead_time.columns:
    raise KeyError(f"Expected column missing from Phase 10 lead-time file: {estimable_col}")

alternative_rows = phase10_lead_time[
    phase10_lead_time["Is_Reference_Mapping"].astype(str).str.lower().isin(["false", "0"])
].copy()

estimable_alternatives = alternative_rows[
    alternative_rows[estimable_col].astype(str).str.lower().isin(["true", "1"])
]

estimated_reduction_value = np.nan
reduction_status = (
    "Not estimable: no validated factory-specific lead-time evidence"
    if estimable_alternatives.empty
    else "Factory-specific estimates require further validation"
)

kpi_lead_time_reduction = pd.DataFrame([{
    "KPI": "Estimated Lead-Time Reduction",
    "Value": estimated_reduction_value,
    "Unit": "Days",
    "Alternative_Options_Checked": len(alternative_rows),
    "Options_With_Estimable_Factory_Lead_Time": len(estimable_alternatives),
    "Status": reduction_status,
    "Interpretation": (
        "No numerical reduction is claimed. Phase 10 marks alternative factory "
        "lead-time effects as not estimable because actual factory assignment "
        "and validated factory-specific outcomes are unavailable."
    )
}])

display(kpi_lead_time_reduction)

,KPI,Value,Unit,Alternative_Options_Checked,Options_With_Estimable_Factory_Lead_Time,Status,Interpretation
0,Estimated Lead-Time Reduction,NaN,Days,60,0,Not estimable: no validated factory-specific l...,No numerical reduction is claimed. Phase 10 ma...


In [18]:
# Profit stability proxy

# Combine the existing margin-stability and gross-profit variability reports

margin_stability = support["margin_stability"].copy()
profit_variability = support["profit_variability"].copy()

required_margin_cols = [
    "Product Name", "Order_Lines", "Mean_Line_Margin_Pct",
    "Std_Line_Margin_Pct", "Margin_IQR_Pct",
    "Margin_Stability_Proxy", "Interpretation"
]
required_profit_cols = [
    "Product Name", "Std_Gross_Profit", "Gross_Profit_IQR",
    "Gross_Profit_CV_Pct"
]

for col in required_margin_cols:
    if col not in margin_stability.columns:
        raise KeyError(f"Missing margin stability column: {col}")
for col in required_profit_cols:
    if col not in profit_variability.columns:
        raise KeyError(f"Missing profit variability column: {col}")

profit_stability = margin_stability[required_margin_cols].merge(
    profit_variability[required_profit_cols],
    on="Product Name",
    how="left",
    validate="one_to_one"
)

profit_stability["Proxy_Status"] = np.where(
    profit_stability["Order_Lines"] < 2,
    "Very limited observations",
    "Historical descriptive proxy"
)
profit_stability["Interpretation_Limit"] = (
    "Higher margin stability proxy indicates lower observed line-margin "
    "standard deviation; it is not a forecast, probability, or causal stability measure."
)

display(profit_stability.head(15))

,Product Name,Order_Lines,Mean_Line_Margin_Pct,Std_Line_Margin_Pct,Margin_IQR_Pct,Margin_Stability_Proxy,Interpretation,Std_Gross_Profit,Gross_Profit_IQR,Gross_Profit_CV_Pct,Proxy_Status,Interpretation_Limit
0,Everlasting Gobstopper,3,80.000000,0.000000e+00,0.000000e+00,1.0,Higher proxy indicates lower observed line-mar...,12.220202,12.000,35.250582,Historical descriptive proxy,Higher margin stability proxy indicates lower ...
1,Fizzy Lifting Drinks,6,60.000000,0.000000e+00,0.000000e+00,1.0,Higher proxy indicates lower observed line-mar...,2.359820,2.250,29.965967,Historical descriptive proxy,Higher margin stability proxy indicates lower ...
2,Fun Dip,3,40.000000,0.000000e+00,0.000000e+00,1.0,Higher proxy indicates lower observed line-mar...,0.916515,0.900,57.282196,Historical descriptive proxy,Higher margin stability proxy indicates lower ...
3,Hair Toffee,4,77.777778,0.000000e+00,0.000000e+00,1.0,Higher proxy indicates lower observed line-mar...,5.250000,2.625,35.294118,Historical descriptive proxy,Higher margin stability proxy indicates lower ...
4,Kazookles,96,7.692308,0.000000e+00,0.000000e+00,1.0,Higher proxy indicates lower observed line-mar...,0.511216,0.750,52.912926,Historical descriptive proxy,Higher margin stability proxy indicates lower ...
5,Laffy Taffy,10,62.311558,0.000000e+00,0.000000e+00,1.0,Higher proxy indicates lower observed line-mar...,2.191057,3.100,65.443745,Historical descriptive proxy,Higher margin stability proxy indicates lower ...
6,Lickable Wallpaper,94,50.000000,0.000000e+00,0.000000e+00,1.0,Higher proxy indicates lower observed line-mar...,24.272767,30.000,58.057001,Historical descriptive proxy,Higher margin stability proxy indicates lower ...
7,Nerds,4,46.666667,0.000000e+00,0.000000e+00,1.0,Higher proxy indicates lower observed line-mar...,0.903696,1.050,51.639778,Historical descriptive proxy,Higher margin stability proxy indicates lower ...
8,SweeTARTS,10,46.666667,0.000000e+00,0.000000e+00,1.0,Higher proxy indicates lower observed line-mar...,1.066719,1.225,37.167901,Historical descriptive proxy,Higher margin stability proxy indicates lower ...
9,Wonka Bar - Fudge Mallows,1818,66.666667,4.243266e-15,0.000000e+00,1.0,Higher proxy indicates lower observed line-mar...,5.273759,7.200,57.779469,Historical descriptive proxy,Higher margin stability proxy indicates lower ...


In [20]:
# Scenario support score

# Measure supported alternative-option coverage without inventing a confidence score

screening = support["phase10_screening"].copy()
unsupported = support["phase10_unsupported"].copy()

alternative_screening = screening[
    screening["Is_Reference_Mapping"].astype(str).str.lower().isin(["false", "0"])
].copy()

if alternative_screening.empty:
    scenario_support = pd.DataFrame(columns=[
        "Product Name", "Alternative_Options", "Supported_Alternative_Options",
        "Scenario_Support_Score_Pct", "Support_Status", "Evidence_Limit"
    ])
else:
    alternative_screening["Is_Supported"] = (
        alternative_screening["Compatibility_Status"].astype(str).str.upper().eq("VERIFIED")
        & alternative_screening["Factory_Specific_Lead_Time_Estimable"].astype(str).str.lower().isin(["true", "1"])
    )

    scenario_support = (
        alternative_screening.groupby("Product Name")
        .agg(
            Alternative_Options=("Factory Option", "nunique"),
            Supported_Alternative_Options=("Is_Supported", "sum")
        )
        .reset_index()
    )

    scenario_support["Scenario_Support_Score_Pct"] = np.where(
        scenario_support["Alternative_Options"] > 0,
        scenario_support["Supported_Alternative_Options"]
        / scenario_support["Alternative_Options"] * 100,
        np.nan
    )

    scenario_support["Support_Status"] = np.where(
        scenario_support["Scenario_Support_Score_Pct"].eq(100),
        "Supported by current screening evidence",
        np.where(
            scenario_support["Scenario_Support_Score_Pct"].gt(0),
            "Partially supported",
            "Unsupported"
        )
    )
    scenario_support["Evidence_Limit"] = (
        "This percentage measures the share of alternative options passing "
        "the documented compatibility and factory-specific lead-time gates. "
        "It is not a probability of success or model confidence."
    )

display(scenario_support)

,Product Name,Alternative_Options,Supported_Alternative_Options,Scenario_Support_Score_Pct,Support_Status,Evidence_Limit
0,Everlasting Gobstopper,4,0,0.0,Unsupported,This percentage measures the share of alternat...
1,Fizzy Lifting Drinks,4,0,0.0,Unsupported,This percentage measures the share of alternat...
2,Fun Dip,4,0,0.0,Unsupported,This percentage measures the share of alternat...
3,Hair Toffee,4,0,0.0,Unsupported,This percentage measures the share of alternat...
4,Kazookles,4,0,0.0,Unsupported,This percentage measures the share of alternat...
5,Laffy Taffy,4,0,0.0,Unsupported,This percentage measures the share of alternat...
6,Lickable Wallpaper,4,0,0.0,Unsupported,This percentage measures the share of alternat...
7,Nerds,4,0,0.0,Unsupported,This percentage measures the share of alternat...
8,SweeTARTS,4,0,0.0,Unsupported,This percentage measures the share of alternat...
9,Wonka Bar - Fudge Mallows,4,0,0.0,Unsupported,This percentage measures the share of alternat...


In [22]:
# Recommendation coverage

# Summarize Phase 10 recommendation and mapping coverage from its saved KPI file

recommendation_coverage = support["phase10_coverage"].copy()

expected_coverage_columns = [
    "Coverage_Metric", "Count", "Interpretation", "Coverage_Percent"
]
for col in expected_coverage_columns:
    if col not in recommendation_coverage.columns:
        raise KeyError(f"Missing Phase 10 coverage column: {col}")

recommendation_coverage["Coverage_Source"] = (
    "Phase 10 task_10_17_coverage_kpis.csv"
)
recommendation_coverage["Interpretation_Limit"] = (
    "Product reference mapping is not proof of historical factory dispatch. "
    "Generated alternative options are screening candidates, not approved options."
)

display(recommendation_coverage)

,Coverage_Metric,Count,Interpretation,Coverage_Percent,Coverage_Source,Interpretation_Limit
0,Source products,15,Products present in the original CSV,100.0,Phase 10 task_10_17_coverage_kpis.csv,Product reference mapping is not proof of hist...
1,Products with reference mapping,15,Products with a Phase 07 reference mapping,100.0,Phase 10 task_10_17_coverage_kpis.csv,Product reference mapping is not proof of hist...
2,Unmapped products,0,Products requiring mapping review,0.0,Phase 10 task_10_17_coverage_kpis.csv,Product reference mapping is not proof of hist...
3,Original source rows represented,10194,Original CSV rows; no row removal,100.0,Phase 10 task_10_17_coverage_kpis.csv,Product reference mapping is not proof of hist...
4,Generated alternative options,60,Other reference factories paired for screening,400.0,Phase 10 task_10_17_coverage_kpis.csv,Product reference mapping is not proof of hist...
5,Supported alternative options,0,No alternative compatibility evidence was supp...,0.0,Phase 10 task_10_17_coverage_kpis.csv,Product reference mapping is not proof of hist...
6,Unsupported alternative options,60,Alternatives lacking compatibility evidence,400.0,Phase 10 task_10_17_coverage_kpis.csv,Product reference mapping is not proof of hist...


In [24]:
# High lead-time risk flags

# Flag products with high historical derived lead time and adequate descriptive sample support

MIN_UNIQUE_ORDERS_FOR_PRODUCT_FLAG = 5
HIGH_LEAD_TIME_QUANTILE = 0.75

lead_time_threshold = product_agg["Mean_Lead_Time_Days"].quantile(
    HIGH_LEAD_TIME_QUANTILE
)

high_lead_time_risk = product_agg[
    [
        "Product Name", "Order_Lines", "Unique_Orders",
        "Mean_Lead_Time_Days", "Median_Lead_Time_Days",
        "Lead_Time_Std_Days", "Lead_Time_IQR_Days"
    ]
].copy()

high_lead_time_risk["Threshold_Days"] = lead_time_threshold
high_lead_time_risk["Sample_Eligible"] = (
    high_lead_time_risk["Unique_Orders"] >= MIN_UNIQUE_ORDERS_FOR_PRODUCT_FLAG
)
high_lead_time_risk["High_Lead_Time_Risk_Flag"] = (
    high_lead_time_risk["Sample_Eligible"]
    & (high_lead_time_risk["Mean_Lead_Time_Days"] >= lead_time_threshold)
)
high_lead_time_risk["Risk_Basis"] = (
    "Product mean derived lead-time at or above the dataset 75th percentile; "
    "minimum 5 distinct orders."
)
high_lead_time_risk["Limitation"] = (
    "Descriptive historical flag only. Derived date differences are not verified "
    "transport or factory transit times."
)

display(
    high_lead_time_risk.sort_values(
        ["High_Lead_Time_Risk_Flag", "Mean_Lead_Time_Days"],
        ascending=[False, False]
    )
)

,Product Name,Order_Lines,Unique_Orders,Mean_Lead_Time_Days,Median_Lead_Time_Days,Lead_Time_Std_Days,Lead_Time_IQR_Days,Threshold_Days,Sample_Eligible,High_Lead_Time_Risk_Flag,Risk_Basis,Limitation
8,SweeTARTS,10,10,1383.500000,1274.0,246.688850,366.50,1361.119149,True,True,Product mean derived lead-time at or above the...,Descriptive historical flag only. Derived date...
5,Laffy Taffy,10,10,1383.100000,1275.0,246.563695,365.00,1361.119149,True,True,Product mean derived lead-time at or above the...,Descriptive historical flag only. Derived date...
3,Hair Toffee,4,4,1455.250000,1455.0,209.876750,361.75,1361.119149,False,False,Product mean derived lead-time at or above the...,Descriptive historical flag only. Derived date...
0,Everlasting Gobstopper,3,3,1394.666667,1273.0,213.336198,185.50,1361.119149,False,False,Product mean derived lead-time at or above the...,Descriptive historical flag only. Derived date...
6,Lickable Wallpaper,94,92,1339.138298,1274.0,288.459855,368.75,1361.119149,True,False,Product mean derived lead-time at or above the...,Descriptive historical flag only. Derived date...
11,Wonka Bar - Nutty Crunch Surprise,1810,1529,1328.915470,1274.0,260.837398,367.00,1361.119149,True,False,Product mean derived lead-time at or above the...,Descriptive historical flag only. Derived date...
12,Wonka Bar - Triple Dazzle Caramel,2015,1677,1325.661042,1274.0,260.524339,367.00,1361.119149,True,False,Product mean derived lead-time at or above the...,Descriptive historical flag only. Derived date...
13,Wonka Bar -Scrumdiddlyumptious,2064,1704,1320.641957,1274.0,260.121265,366.00,1361.119149,True,False,Product mean derived lead-time at or above the...,Descriptive historical flag only. Derived date...
10,Wonka Bar - Milk Chocolate,2137,1768,1316.764623,1274.0,263.692704,367.00,1361.119149,True,False,Product mean derived lead-time at or above the...,Descriptive historical flag only. Derived date...
9,Wonka Bar - Fudge Mallows,1818,1527,1314.242024,1274.0,264.102825,367.00,1361.119149,True,False,Product mean derived lead-time at or above the...,Descriptive historical flag only. Derived date...


In [26]:
# Low-margin risk flags

# Flag products at or below break-even margin and separately identify lower-quartile margins

LOW_MARGIN_BREAK_EVEN_PCT = 0.0
LOW_MARGIN_QUANTILE = 0.25
MIN_UNIQUE_ORDERS_FOR_MARGIN_FLAG = 5

margin_watch_threshold = product_agg["Gross_Margin_Pct"].quantile(
    LOW_MARGIN_QUANTILE
)

low_margin_risk = product_agg[
    [
        "Product Name", "Order_Lines", "Unique_Orders",
        "Total_Sales", "Total_Gross_Profit", "Gross_Margin_Pct"
    ]
].copy()

low_margin_risk["Break_Even_Risk_Flag"] = (
    low_margin_risk["Gross_Margin_Pct"] <= LOW_MARGIN_BREAK_EVEN_PCT
)
low_margin_risk["Lower_Quartile_Margin_Flag"] = (
    low_margin_risk["Gross_Margin_Pct"] <= margin_watch_threshold
)
low_margin_risk["Sample_Eligible"] = (
    low_margin_risk["Unique_Orders"] >= MIN_UNIQUE_ORDERS_FOR_MARGIN_FLAG
)
low_margin_risk["Low_Margin_Risk_Flag"] = (
    low_margin_risk["Sample_Eligible"]
    & (
        low_margin_risk["Break_Even_Risk_Flag"]
        | low_margin_risk["Lower_Quartile_Margin_Flag"]
    )
)
low_margin_risk["Lower_Quartile_Threshold_Pct"] = margin_watch_threshold
low_margin_risk["Interpretation"] = (
    "Break-even and dataset-relative margin flags are historical screening signals, "
    "not company-approved margin targets."
)

display(
    low_margin_risk.sort_values(
        ["Low_Margin_Risk_Flag", "Gross_Margin_Pct"],
        ascending=[False, True]
    )
)

,Product Name,Order_Lines,Unique_Orders,Total_Sales,Total_Gross_Profit,Gross_Margin_Pct,Break_Even_Risk_Flag,Lower_Quartile_Margin_Flag,Sample_Eligible,Low_Margin_Risk_Flag,Lower_Quartile_Threshold_Pct,Interpretation
4,Kazookles,96,94,1205.75,92.75,7.692308,False,True,True,True,48.333333,Break-even and dataset-relative margin flags a...
8,SweeTARTS,10,10,61.50,28.70,46.666667,False,True,True,True,48.333333,Break-even and dataset-relative margin flags a...
2,Fun Dip,3,3,12.00,4.80,40.000000,False,True,False,False,48.333333,Break-even and dataset-relative margin flags a...
7,Nerds,4,4,15.00,7.00,46.666667,False,True,False,False,48.333333,Break-even and dataset-relative margin flags a...
6,Lickable Wallpaper,94,92,7860.00,3930.00,50.000000,False,False,True,False,48.333333,Break-even and dataset-relative margin flags a...
14,Wonka Gum,120,118,597.50,310.70,52.000000,False,False,True,False,48.333333,Break-even and dataset-relative margin flags a...
1,Fizzy Lifting Drinks,6,6,78.75,47.25,60.000000,False,False,True,False,48.333333,Break-even and dataset-relative margin flags a...
5,Laffy Taffy,10,10,53.73,33.48,62.311558,False,False,True,False,48.333333,Break-even and dataset-relative margin flags a...
10,Wonka Bar - Milk Chocolate,2137,1768,26867.75,17443.37,64.923077,False,False,True,False,48.333333,Break-even and dataset-relative margin flags a...
12,Wonka Bar - Triple Dazzle Caramel,2015,1677,28485.00,18610.20,65.333333,False,False,True,False,48.333333,Break-even and dataset-relative margin flags a...


In [28]:
# Region performance risk flags

# Assess region-level historical margin and derived lead-time signals

MIN_UNIQUE_ORDERS_FOR_REGION_FLAG = 5
REGION_LEAD_TIME_QUANTILE = 0.75

region_lead_time_threshold = region_agg["Mean_Lead_Time_Days"].quantile(
    REGION_LEAD_TIME_QUANTILE
)
region_margin_threshold = region_agg["Gross_Margin_Pct"].quantile(0.25)

region_performance_risk = region_agg.copy()

region_performance_risk["Sample_Eligible"] = (
    region_performance_risk["Unique_Orders"] >= MIN_UNIQUE_ORDERS_FOR_REGION_FLAG
)
region_performance_risk["Low_or_Negative_Margin_Flag"] = (
    region_performance_risk["Gross_Margin_Pct"] <= region_margin_threshold
)
region_performance_risk["High_Derived_Lead_Time_Flag"] = (
    region_performance_risk["Mean_Lead_Time_Days"] >= region_lead_time_threshold
)
region_performance_risk["Region_Performance_Risk_Flag"] = (
    region_performance_risk["Sample_Eligible"]
    & (
        region_performance_risk["Low_or_Negative_Margin_Flag"]
        | region_performance_risk["High_Derived_Lead_Time_Flag"]
    )
)
region_performance_risk["Margin_Threshold_Pct"] = region_margin_threshold
region_performance_risk["Lead_Time_Threshold_Days"] = region_lead_time_threshold
region_performance_risk["Interpretation_Limit"] = (
    "Only four regions are represented. These are descriptive comparisons; "
    "they do not establish the cause of regional performance differences."
)

display(region_performance_risk)

,Region,Order_Lines,Unique_Orders,Total_Sales,Total_Gross_Profit,Total_Units,Mean_Lead_Time_Days,Median_Lead_Time_Days,Lead_Time_Std_Days,Gross_Margin_Pct,Sample_Eligible,Low_or_Negative_Margin_Flag,High_Derived_Lead_Time_Flag,Region_Performance_Risk_Flag,Margin_Threshold_Pct,Lead_Time_Threshold_Days,Interpretation_Limit
0,Atlantic,2986,2476,41197.24,26973.70,11159,1322.745144,1274.0,256.541810,65.474532,True,True,False,True,65.750273,1322.831663,Only four regions are represented. These are d...
1,Gulf,1620,1350,22247.26,14700.67,6209,1311.374691,1274.0,264.727855,66.078564,True,False,False,False,65.750273,1322.831663,Only four regions are represented. These are d...
2,Interior,2335,1979,32037.60,21282.49,8820,1323.091221,1274.0,262.693597,66.429726,True,False,True,True,65.750273,1322.831663,Only four regions are represented. These are d...
3,Pacific,3253,2744,46301.53,30485.94,12466,1322.194897,1274.0,266.470647,65.842187,True,False,False,False,65.750273,1322.831663,Only four regions are represented. These are d...


In [30]:
# Low sample-size risk flags

# Classify product and region evidence by distinct order count

LOW_SAMPLE_ORDER_THRESHOLD = 10
LIMITED_SAMPLE_ORDER_THRESHOLD = 30

def classify_sample_support(order_count):
    if pd.isna(order_count):
        return "Unknown"
    if order_count <= LOW_SAMPLE_ORDER_THRESHOLD:
        return "Low sample"
    if order_count < LIMITED_SAMPLE_ORDER_THRESHOLD:
        return "Limited sample"
    return "Higher sample support"

product_sample_risk = product_agg[
    ["Product Name", "Order_Lines", "Unique_Orders"]
].copy()
product_sample_risk["Sample_Support_Category"] = (
    product_sample_risk["Unique_Orders"].apply(classify_sample_support)
)
product_sample_risk["Low_Sample_Size_Risk_Flag"] = (
    product_sample_risk["Unique_Orders"] <= LOW_SAMPLE_ORDER_THRESHOLD
)
product_sample_risk["Threshold_Explanation"] = (
    "Screening rule: 10 or fewer distinct orders = low sample; "
    "11–29 = limited sample; 30 or more = higher sample support. "
    "These are project analysis thresholds, not industry standards."
)

region_sample_risk = region_agg[
    ["Region", "Order_Lines", "Unique_Orders"]
].copy()
region_sample_risk["Sample_Support_Category"] = (
    region_sample_risk["Unique_Orders"].apply(classify_sample_support)
)
region_sample_risk["Low_Sample_Size_Risk_Flag"] = (
    region_sample_risk["Unique_Orders"] <= LOW_SAMPLE_ORDER_THRESHOLD
)

display(product_sample_risk.sort_values("Unique_Orders"))
display(region_sample_risk.sort_values("Unique_Orders"))

,Product Name,Order_Lines,Unique_Orders,Sample_Support_Category,Low_Sample_Size_Risk_Flag,Threshold_Explanation
0,Everlasting Gobstopper,3,3,Low sample,True,Screening rule: 10 or fewer distinct orders = ...
2,Fun Dip,3,3,Low sample,True,Screening rule: 10 or fewer distinct orders = ...
3,Hair Toffee,4,4,Low sample,True,Screening rule: 10 or fewer distinct orders = ...
7,Nerds,4,4,Low sample,True,Screening rule: 10 or fewer distinct orders = ...
1,Fizzy Lifting Drinks,6,6,Low sample,True,Screening rule: 10 or fewer distinct orders = ...
5,Laffy Taffy,10,10,Low sample,True,Screening rule: 10 or fewer distinct orders = ...
8,SweeTARTS,10,10,Low sample,True,Screening rule: 10 or fewer distinct orders = ...
6,Lickable Wallpaper,94,92,Higher sample support,False,Screening rule: 10 or fewer distinct orders = ...
4,Kazookles,96,94,Higher sample support,False,Screening rule: 10 or fewer distinct orders = ...
14,Wonka Gum,120,118,Higher sample support,False,Screening rule: 10 or fewer distinct orders = ...


,Region,Order_Lines,Unique_Orders,Sample_Support_Category,Low_Sample_Size_Risk_Flag
1,Gulf,1620,1350,Higher sample support,False
2,Interior,2335,1979,Higher sample support,False
0,Atlantic,2986,2476,Higher sample support,False
3,Pacific,3253,2744,Higher sample support,False


In [32]:
# Model uncertainty notes

# Compare saved model metadata with model comparison and cross-validation evidence

model_comparison = support["model_comparison"].copy()
cross_validation = support["cross_validation"].copy()
predictions = support["predictions"].copy()
lead_time_ranges = support["lead_time_ranges"].copy()

metadata_metrics = {
    "Test_MAE": model_metadata.get("test_MAE"),
    "Test_RMSE": model_metadata.get("test_RMSE"),
    "Test_R2": model_metadata.get("test_R2"),
}

selected_model_name = model_metadata.get("selected_model")
comparison_selected = model_comparison[
    model_comparison["Model"].astype(str).str.casefold()
    == str(selected_model_name).casefold()
]

if comparison_selected.empty:
    comparison_metrics = {
        "Test_MAE": np.nan, "Test_RMSE": np.nan, "Test_R2": np.nan
    }
    comparison_status = "Selected model not found in final comparison file"
else:
    row = comparison_selected.iloc[0]
    comparison_metrics = {
        "Test_MAE": row.get("Test_MAE"),
        "Test_RMSE": row.get("Test_RMSE"),
        "Test_R2": row.get("Test_R2"),
    }
    comparison_status = "Found in final comparison file"

uncertainty_notes = [
    {
        "Issue": "Saved metadata versus final comparison metrics",
        "Evidence": (
            f"Metadata MAE={metadata_metrics['Test_MAE']}, "
            f"RMSE={metadata_metrics['Test_RMSE']}, R2={metadata_metrics['Test_R2']}; "
            f"comparison MAE={comparison_metrics['Test_MAE']}, "
            f"RMSE={comparison_metrics['Test_RMSE']}, R2={comparison_metrics['Test_R2']}."
        ),
        "Status": "Discrepancy requires reconciliation",
        "Impact": "Do not publish one metric set as the definitive model result until traced."
    },
    {
        "Issue": "Cross-validation model identity",
        "Evidence": (
            "Cross-validation file contains: "
            + ", ".join(cross_validation["Model"].astype(str).tolist())
        ),
        "Status": (
            "Tuned Random Forest cross-validation row not present"
            if not cross_validation["Model"].astype(str).str.casefold().eq(
                str(selected_model_name).casefold()
            ).any()
            else "Selected model appears in cross-validation file"
        ),
        "Impact": "Cross-validation evidence does not directly validate the selected tuned model."
    },
    {
        "Issue": "Lead-time target interpretation",
        "Evidence": (
            "Target variable is Lead_Time_Days derived from Order Date and Ship Date. "
            "Observed source date differences are unusually long."
        ),
        "Status": "Interpretation limitation",
        "Impact": "Predictions must not be presented as verified delivery or transit-time forecasts."
    },
    {
        "Issue": "Predictor availability and leakage review",
        "Evidence": (
            "Metadata lists Sales, Gross Profit, Cost, Profit_Margin_Pct, "
            "Sales_Per_Unit, Cost_Per_Unit and Profit_Per_Unit as predictors."
        ),
        "Status": "Prediction-time availability needs review",
        "Impact": (
            "Confirm these fields are available at the intended prediction point; "
            "otherwise model performance may not represent a real operational forecast."
        )
    },
    {
        "Issue": "Factory reassignment inference",
        "Evidence": (
            "Phase 10 has no verified actual factory dispatch or factory-specific "
            "lead-time estimates for alternatives."
        ),
        "Status": "Not supported",
        "Impact": "The model does not establish lead-time reduction from changing factories."
    }
]

model_uncertainty_report = pd.DataFrame(uncertainty_notes)

display(model_uncertainty_report)
display(model_comparison)
display(cross_validation)

,Issue,Evidence,Status,Impact
0,Saved metadata versus final comparison metrics,"Metadata MAE=95.97949651478011, RMSE=120.73823...",Discrepancy requires reconciliation,Do not publish one metric set as the definitiv...
1,Cross-validation model identity,"Cross-validation file contains: Random Forest,...",Tuned Random Forest cross-validation row not p...,Cross-validation evidence does not directly va...
2,Lead-time target interpretation,Target variable is Lead_Time_Days derived from...,Interpretation limitation,Predictions must not be presented as verified ...
3,Predictor availability and leakage review,"Metadata lists Sales, Gross Profit, Cost, Prof...",Prediction-time availability needs review,Confirm these fields are available at the inte...
4,Factory reassignment inference,Phase 10 has no verified actual factory dispat...,Not supported,The model does not establish lead-time reducti...


,Model,Test_MAE,Test_RMSE,Test_R2
0,Tuned Random Forest,139.670688,163.675428,0.619499
1,Random Forest,136.586130,163.904306,0.618435
2,Gradient Boosting,163.302233,171.306667,0.583191
3,Linear Regression,166.414841,180.784694,0.535793


,Model,Mean_CV_MAE,Mean_CV_RMSE,Mean_CV_R2
0,Random Forest,135.678649,163.178470,0.610839
1,Gradient Boosting,161.891102,170.029563,0.577509
2,Linear Regression,164.263762,179.851424,0.527213


In [34]:
# KPI dashboard data

# Assemble a consistent KPI table for dashboard use

dashboard_rows = [
    {
        "Metric": "Total Sales",
        "Value": total_sales,
        "Unit": "Source currency units",
        "Scope": "All source order lines",
        "Status": "Calculated",
        "Interpretation": "Historical sum of Sales."
    },
    {
        "Metric": "Total Gross Profit",
        "Value": total_gross_profit,
        "Unit": "Source currency units",
        "Scope": "All source order lines",
        "Status": "Calculated",
        "Interpretation": "Historical sum of the Gross Profit source field."
    },
    {
        "Metric": "Overall Gross Profit Margin",
        "Value": overall_margin_pct,
        "Unit": "%",
        "Scope": "Total Gross Profit / Total Sales",
        "Status": "Calculated" if pd.notna(overall_margin_pct) else "Not calculable",
        "Interpretation": "Weighted overall gross margin."
    },
    {
        "Metric": "Average Derived Lead Time",
        "Value": average_derived_lead_time,
        "Unit": "Days",
        "Scope": "Valid parsed date pairs",
        "Status": "Historical derived measure",
        "Interpretation": "Not verified transit duration."
    },
    {
        "Metric": "Median Derived Lead Time",
        "Value": median_derived_lead_time,
        "Unit": "Days",
        "Scope": "Valid parsed date pairs",
        "Status": "Historical derived measure",
        "Interpretation": "Not verified transit duration."
    },
    {
        "Metric": "Derived Lead-Time Standard Deviation",
        "Value": lead_time_std,
        "Unit": "Days",
        "Scope": "Valid parsed date pairs",
        "Status": "Historical descriptive measure",
        "Interpretation": "Sample standard deviation; ddof=1."
    },
    {
        "Metric": "Derived Lead-Time IQR",
        "Value": lead_time_iqr,
        "Unit": "Days",
        "Scope": "Valid parsed date pairs",
        "Status": "Historical descriptive measure",
        "Interpretation": "75th percentile minus 25th percentile."
    },
    {
        "Metric": "Estimated Lead-Time Reduction",
        "Value": np.nan,
        "Unit": "Days",
        "Scope": "Alternative factory scenarios",
        "Status": "Not estimable",
        "Interpretation": "No validated factory-specific lead-time effect is available."
    },
    {
        "Metric": "Supported Alternative Option Coverage",
        "Value": (
            alternative_screening["Is_Supported"].mean() * 100
            if len(alternative_screening) else np.nan
        ),
        "Unit": "%",
        "Scope": "Phase 10 alternative factory options",
        "Status": "Evidence coverage, not confidence",
        "Interpretation": "Share of alternatives passing compatibility and lead-time evidence gates."
    }
]

kpi_dashboard_data = pd.DataFrame(dashboard_rows)

display(kpi_dashboard_data)

,Metric,Value,Unit,Scope,Status,Interpretation
0,Total Sales,141783.630000,Source currency units,All source order lines,Calculated,Historical sum of Sales.
1,Total Gross Profit,93442.800000,Source currency units,All source order lines,Calculated,Historical sum of the Gross Profit source field.
2,Overall Gross Profit Margin,65.905211,%,Total Gross Profit / Total Sales,Calculated,Weighted overall gross margin.
3,Average Derived Lead Time,1320.841868,Days,Valid parsed date pairs,Historical derived measure,Not verified transit duration.
4,Median Derived Lead Time,1274.000000,Days,Valid parsed date pairs,Historical derived measure,Not verified transit duration.
5,Derived Lead-Time Standard Deviation,262.444892,Days,Valid parsed date pairs,Historical descriptive measure,Sample standard deviation; ddof=1.
6,Derived Lead-Time IQR,367.000000,Days,Valid parsed date pairs,Historical descriptive measure,75th percentile minus 25th percentile.
7,Estimated Lead-Time Reduction,NaN,Days,Alternative factory scenarios,Not estimable,No validated factory-specific lead-time effect...
8,Supported Alternative Option Coverage,0.000000,%,Phase 10 alternative factory options,"Evidence coverage, not confidence",Share of alternatives passing compatibility an...


In [36]:
# Risk register

# Create a consolidated risk register distinguishing historical signals from verified operational risks

risk_register_rows = []

# Product-level high derived lead-time flags
for _, row in high_lead_time_risk[
    high_lead_time_risk["High_Lead_Time_Risk_Flag"]
].iterrows():
    risk_register_rows.append({
        "Risk_ID": f"LT-{len(risk_register_rows) + 1:03d}",
        "Category": "High derived lead time",
        "Scope": f"Product: {row['Product Name']}",
        "Indicator": "Mean derived lead-time days",
        "Threshold": row["Threshold_Days"],
        "Observed_Value": row["Mean_Lead_Time_Days"],
        "Severity": "Review",
        "Evidence_Type": "Historical descriptive signal",
        "Evidence": row["Risk_Basis"],
        "Limitation": row["Limitation"],
        "Suggested_Action_or_Data_Needed": (
            "Validate source date definitions and obtain actual order, dispatch, "
            "and delivery event timestamps."
        )
    })

# Product-level low margin flags
for _, row in low_margin_risk[
    low_margin_risk["Low_Margin_Risk_Flag"]
].iterrows():
    risk_register_rows.append({
        "Risk_ID": f"LM-{len(risk_register_rows) + 1:03d}",
        "Category": "Low historical gross margin",
        "Scope": f"Product: {row['Product Name']}",
        "Indicator": "Historical gross margin",
        "Threshold": margin_watch_threshold,
        "Observed_Value": row["Gross_Margin_Pct"],
        "Severity": (
            "High review priority"
            if row["Break_Even_Risk_Flag"] else "Review"
        ),
        "Evidence_Type": "Historical descriptive signal",
        "Evidence": (
            "Product gross profit divided by product sales; "
            "break-even and lower-quartile flags applied."
        ),
        "Limitation": (
            "Does not include unrecorded overhead, future prices, or "
            "unobserved cost changes."
        ),
        "Suggested_Action_or_Data_Needed": (
            "Review product pricing, recorded cost components, discounts, "
            "and current contribution-margin definitions."
        )
    })

# Region-level performance flags
for _, row in region_performance_risk[
    region_performance_risk["Region_Performance_Risk_Flag"]
].iterrows():
    risk_register_rows.append({
        "Risk_ID": f"RG-{len(risk_register_rows) + 1:03d}",
        "Category": "Region performance signal",
        "Scope": f"Region: {row['Region']}",
        "Indicator": "Historical margin and/or mean derived lead time",
        "Threshold": (
            f"Margin <= {region_margin_threshold:.4f}% or derived lead time "
            f">= {region_lead_time_threshold:.2f} days"
        ),
        "Observed_Value": (
            f"Margin={row['Gross_Margin_Pct']:.4f}%; "
            f"mean derived lead time={row['Mean_Lead_Time_Days']:.2f} days"
        ),
        "Severity": "Review",
        "Evidence_Type": "Historical descriptive signal",
        "Evidence": "Region-level historical aggregate.",
        "Limitation": (
            "Only four regions are present; this does not establish causation "
            "or an operational failure."
        ),
        "Suggested_Action_or_Data_Needed": (
            "Validate regional order mix, event dates, customer service measures, "
            "and operational process data."
        )
    })

# Low sample-size flags
for _, row in product_sample_risk[
    product_sample_risk["Low_Sample_Size_Risk_Flag"]
].iterrows():
    risk_register_rows.append({
        "Risk_ID": f"SS-{len(risk_register_rows) + 1:03d}",
        "Category": "Low sample size",
        "Scope": f"Product: {row['Product Name']}",
        "Indicator": "Distinct historical orders",
        "Threshold": LOW_SAMPLE_ORDER_THRESHOLD,
        "Observed_Value": row["Unique_Orders"],
        "Severity": "Evidence limitation",
        "Evidence_Type": "Sample-size limitation",
        "Evidence": row["Threshold_Explanation"],
        "Limitation": "Small samples make product-level comparisons less stable.",
        "Suggested_Action_or_Data_Needed": (
            "Collect more distinct orders before making product-level decisions."
        )
    })

# Global model evidence and factory reassignment limitations
risk_register_rows.extend([
    {
        "Risk_ID": "MD-001",
        "Category": "Model metric inconsistency",
        "Scope": "Lead-time prediction model",
        "Indicator": "Metadata versus final comparison test metrics",
        "Threshold": "Metrics should reconcile to the same model run and test split",
        "Observed_Value": "Metadata and comparison metrics differ",
        "Severity": "High - resolve before final model claims",
        "Evidence_Type": "Model validation issue",
        "Evidence": "Saved metadata and ML_Final_Model_Comparison.csv report different test metrics.",
        "Limitation": "The authoritative metric set has not been established.",
        "Suggested_Action_or_Data_Needed": (
            "Re-run the saved evaluation pipeline and reconcile model version, "
            "test split, and exported metric files."
        )
    },
    {
        "Risk_ID": "FC-001",
        "Category": "Factory reassignment evidence gap",
        "Scope": "Alternative factory scenarios",
        "Indicator": "Verified alternative options",
        "Threshold": "Verified compatibility and factory-specific outcome evidence required",
        "Observed_Value": int(len(unsupported)),
        "Severity": "High - scenario claims blocked",
        "Evidence_Type": "Missing operational evidence",
        "Evidence": "Phase 10 reports unsupported alternative factory options.",
        "Limitation": "No verified dispatch, capacity, inventory, production-time, freight, or factory-specific cost data.",
        "Suggested_Action_or_Data_Needed": (
            "Obtain verified product-factory compatibility, actual dispatch history, "
            "capacity, inventory, production time, and cost data."
        )
    },
    {
        "Risk_ID": "DT-001",
        "Category": "Date-derived lead-time interpretation",
        "Scope": "All source records",
        "Indicator": "Order Date-to-Ship Date difference",
        "Threshold": "Validate date semantics before operational interpretation",
        "Observed_Value": (
            f"Observed range: {valid_lead_times.min():.0f} to "
            f"{valid_lead_times.max():.0f} days"
            if len(valid_lead_times) else "No valid date pairs"
        ),
        "Severity": "High - interpretation limitation",
        "Evidence_Type": "Source date-derived measure",
        "Evidence": "Derived date differences are unusually long.",
        "Limitation": "May not represent actual transit or production lead time.",
        "Suggested_Action_or_Data_Needed": (
            "Confirm date formats, date definitions, and event timestamps with the source owner."
        )
    }
])

risk_register = pd.DataFrame(risk_register_rows)

if risk_register.empty:
    risk_register = pd.DataFrame(columns=[
        "Risk_ID", "Category", "Scope", "Indicator", "Threshold",
        "Observed_Value", "Severity", "Evidence_Type", "Evidence",
        "Limitation", "Suggested_Action_or_Data_Needed"
    ])

display(risk_register)

,Risk_ID,Category,Scope,Indicator,Threshold,Observed_Value,Severity,Evidence_Type,Evidence,Limitation,Suggested_Action_or_Data_Needed
0,LT-001,High derived lead time,Product: Laffy Taffy,Mean derived lead-time days,1361.119149,1383.1,Review,Historical descriptive signal,Product mean derived lead-time at or above the...,Descriptive historical flag only. Derived date...,Validate source date definitions and obtain ac...
1,LT-002,High derived lead time,Product: SweeTARTS,Mean derived lead-time days,1361.119149,1383.5,Review,Historical descriptive signal,Product mean derived lead-time at or above the...,Descriptive historical flag only. Derived date...,Validate source date definitions and obtain ac...
2,LM-003,Low historical gross margin,Product: Kazookles,Historical gross margin,48.333333,7.692308,Review,Historical descriptive signal,Product gross profit divided by product sales;...,"Does not include unrecorded overhead, future p...","Review product pricing, recorded cost componen..."
3,LM-004,Low historical gross margin,Product: SweeTARTS,Historical gross margin,48.333333,46.666667,Review,Historical descriptive signal,Product gross profit divided by product sales;...,"Does not include unrecorded overhead, future p...","Review product pricing, recorded cost componen..."
4,RG-005,Region performance signal,Region: Atlantic,Historical margin and/or mean derived lead time,Margin <= 65.7503% or derived lead time >= 132...,Margin=65.4745%; mean derived lead time=1322.7...,Review,Historical descriptive signal,Region-level historical aggregate.,Only four regions are present; this does not e...,"Validate regional order mix, event dates, cust..."
5,RG-006,Region performance signal,Region: Interior,Historical margin and/or mean derived lead time,Margin <= 65.7503% or derived lead time >= 132...,Margin=66.4297%; mean derived lead time=1323.0...,Review,Historical descriptive signal,Region-level historical aggregate.,Only four regions are present; this does not e...,"Validate regional order mix, event dates, cust..."
6,SS-007,Low sample size,Product: Everlasting Gobstopper,Distinct historical orders,10,3,Evidence limitation,Sample-size limitation,Screening rule: 10 or fewer distinct orders = ...,Small samples make product-level comparisons l...,Collect more distinct orders before making pro...
7,SS-008,Low sample size,Product: Fizzy Lifting Drinks,Distinct historical orders,10,6,Evidence limitation,Sample-size limitation,Screening rule: 10 or fewer distinct orders = ...,Small samples make product-level comparisons l...,Collect more distinct orders before making pro...
8,SS-009,Low sample size,Product: Fun Dip,Distinct historical orders,10,3,Evidence limitation,Sample-size limitation,Screening rule: 10 or fewer distinct orders = ...,Small samples make product-level comparisons l...,Collect more distinct orders before making pro...
9,SS-010,Low sample size,Product: Hair Toffee,Distinct historical orders,10,4,Evidence limitation,Sample-size limitation,Screening rule: 10 or fewer distinct orders = ...,Small samples make product-level comparisons l...,Collect more distinct orders before making pro...


In [44]:
# Save all outputs

# Save all  KPI, risk analysis, and dashboard outputs

output_tables = {
    "kpi_total_sales.csv": kpi_total_sales,
    "kpi_gross_profit.csv": kpi_gross_profit,
    "kpi_profit_margin.csv": kpi_profit_margin,
    "kpi_average_derived_lead_time.csv": kpi_average_lead_time,
    "kpi_median_derived_lead_time.csv": kpi_median_lead_time,
    "kpi_lead_time_variability.csv": kpi_lead_time_variability,
    "kpi_estimated_lead_time_reduction.csv": kpi_lead_time_reduction,
    "profit_stability_proxy.csv": profit_stability,
    "scenario_support_score.csv": scenario_support,
    "recommendation_coverage.csv": recommendation_coverage,
    "high_derived_lead_time_risk.csv": high_lead_time_risk,
    "low_margin_risk.csv": low_margin_risk,
    "region_performance_risk.csv": region_performance_risk,
    "low_sample_size_risk.csv": product_sample_risk,
    "region_sample_size_risk.csv": region_sample_risk,
    "model_uncertainty_report.csv": model_uncertainty_report,
    "kpi_dashboard_data.csv": kpi_dashboard_data,
    "risk_register.csv": risk_register
}

# Export CSV files
for filename, table in output_tables.items():
    table.to_csv(OUTPUT_DIR / filename, index=False)

# Export combined Excel workbook
with pd.ExcelWriter(OUTPUT_DIR / "Phase_11_KPI_Risk_Analysis.xlsx") as writer:
    for filename, table in output_tables.items():
        sheet_name = filename.replace(".csv", "")[:31]
        table.to_excel(writer, sheet_name=sheet_name, index=False)

    region_sample_risk.to_excel(
        writer, sheet_name="region_sample_size", index=False
    )

# Verify exported files
missing_exports = [
    filename for filename in output_tables
    if not (OUTPUT_DIR / filename).exists()
]

excel_file = OUTPUT_DIR / "KPI_Risk_Analysis.xlsx"

print("CSV files exported:", len(output_tables) - len(missing_exports))
print("Excel workbook created:", excel_file.exists())
print("Missing exports:", missing_exports)
print("Output folder:", OUTPUT_DIR)

if not missing_exports and excel_file.exists():
    print("PHASE 11 OUTPUT EXPORT COMPLETED SUCCESSFULLY.")
else:
    print("WARNING: Some exports are missing. Check the errors above.")

CSV files exported: 18
Excel workbook created: False
Missing exports: []
Output folder: C:\Users\hp\Nassau_Candy_Final_Submission\reports\kpi_risk_analysis


In [46]:
# Validate  exports and confirm that the original source file is unchanged

source_hash_after = hashlib.sha256(RAW_FILE.read_bytes()).hexdigest()

integrity_checks = [
    {
        "Check": "Original source SHA-256 unchanged",
        "Passed": source_hash_before == source_hash_after == EXPECTED_SOURCE_SHA256,
        "Detail": f"Before={source_hash_before}; After={source_hash_after}"
    },
    {
        "Check": "Original source row count preserved in memory",
        "Passed": len(df) == 10194,
        "Detail": f"Rows read={len(df)}"
    },
    {
        "Check": "Original source columns preserved in memory",
        "Passed": list(df.columns) == required_source_columns,
        "Detail": f"Columns read={len(df.columns)}"
    },
    {
        "Check": "Original source date strings retained",
        "Passed": (
            df["Order Date"].equals(pd.read_csv(RAW_FILE)["Order Date"])
            and df["Ship Date"].equals(pd.read_csv(RAW_FILE)["Ship Date"])
        ),
        "Detail": "Date parsing was performed only in separate derived columns."
    },
    {
        "Check": "Lead-time reduction not fabricated",
        "Passed": pd.isna(kpi_lead_time_reduction.loc[0, "Value"]),
        "Detail": kpi_lead_time_reduction.loc[0, "Status"]
    },
    {
        "Check": "Scenario support score is evidence-based",
        "Passed": (
            scenario_support.empty
            or scenario_support["Scenario_Support_Score_Pct"].between(0, 100).all()
        ),
        "Detail": "Score is the percentage of alternatives passing documented evidence gates."
    },
    {
        "Check": "All expected output files exist",
        "Passed": all((OUTPUT_DIR / name).exists() for name in output_tables),
        "Detail": f"Expected CSV files={len(output_tables)}"
    },
    {
        "Check": "Risk register created",
        "Passed": (OUTPUT_DIR / "risk_register.csv").exists(),
        "Detail": f"Risk records={len(risk_register)}"
    },
    {
        "Check": "Dashboard KPI output created",
        "Passed": (OUTPUT_DIR / "kpi_dashboard_data.csv").exists(),
        "Detail": f"Dashboard metrics={len(kpi_dashboard_data)}"
    },
]

phase11_integrity_report = pd.DataFrame(integrity_checks)
display(phase11_integrity_report)

phase11_integrity_report.to_csv(
    OUTPUT_DIR / "phase11_integrity_report.csv",
    index=False
)

if not phase11_integrity_report["Passed"].all():
    print("WARNING: One or more integrity checks failed. Review before submission.")
else:
    print("All  integrity checks passed.")

,Check,Passed,Detail
0,Original source SHA-256 unchanged,True,Before=6ae9208e77ce410e90073b19813d29f89853ce9...
1,Original source row count preserved in memory,True,Rows read=10194
2,Original source columns preserved in memory,True,Columns read=18
3,Original source date strings retained,True,Date parsing was performed only in separate de...
4,Lead-time reduction not fabricated,True,Not estimable: no validated factory-specific l...
5,Scenario support score is evidence-based,True,Score is the percentage of alternatives passin...
6,All expected output files exist,True,Expected CSV files=18
7,Risk register created,True,Risk records=16
8,Dashboard KPI output created,True,Dashboard metrics=9


All  integrity checks passed.
